# Organização do Ambiente

Este notebook prepara a infraestrutura utilizada pelo projeto.

Aqui são definidos o catálogo, os schemas das camadas Bronze, Silver e Gold e o Volume utilizado como Landing Zone para os arquivos de entrada.

In [0]:
# Configurações utilizadas em todos os notebooks do pipeline
catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Caminho do Volume onde os arquivos CSV brutos são disponibilizados
landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: olist_medallion
bronze_schema: olist_medallion.bronze
silver_schema: olist_medallion.silver
gold_schema: olist_medallion.gold
landing_path: /Volumes/olist_medallion/bronze/Landing


## Criação da infraestrutura

São criados o catálogo, os schemas correspondentes às três camadas da Arquitetura Medalhão e o Volume `Landing`, utilizado para armazenar os dados brutos de entrada.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catalog, schemas e volume prontos.")

Catalog, schemas e volume prontos.


## Validação dos arquivos de entrada

Antes da execução da camada Bronze, é verificado se todos os cinco arquivos CSV esperados estão disponíveis na Landing Zone.

Essa validação evita a execução parcial do pipeline caso algum arquivo obrigatório esteja ausente.

In [0]:
arquivos_esperados = [
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in arquivos_esperados if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos esperados estão na landing zone:
  - cotacao_bcb.json (26.4 KB)
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)
